# Fast about_subalgebras

## Prompt

In the 'finalg' module, the function 'about_subalgebras' in 'about_subalgebras.py' finds all proper subalgebras of an algebra, as a list of algebras. Then it partitions them into sublists of algebras that are isomorphic to each other. FInally, it prints a summary of the resulting list of lists ('partitions') of subalgebras, and returns the list of lists of algebra. However, it is slow for large algebras, such as A_5. Can you suggest a way to speed it up? If so, please write a faster version of the function.

## Answer

A5 already exists as a stored example. Let's load it and profile the current implementation.Good, I can load A5 directly from the JSON file.Good — 25s is already a massive improvement (the old code would take longer than the age of the universe on A5), but let's see if we can push it further before finalizing.Let's test the full pipeline end-to-end with all fixes combined (subgroup enumeration + fast op lookup + fast_isomorphic-based partitioning) to see the real-world result on A5.I dug into this and found that the real bottleneck isn't actually in `about_subalgebras.py` itself — it's one layer down, in `Magma.proper_subalgebras()` / `closed_subsets_of_elements()` (in `magma.py`), which `about_subalgebras()` calls first. There's a second, smaller bottleneck in `about_subalgebras.py` itself. Both need fixing to make A₅ tractable.

## Bottleneck #1 (the big one): `closed_subsets_of_elements`

```python
for i in range_:   # divisors of n, e.g. [2,3,4,5,6,10,12,15,20,30] for n=60
    for combo in it.combinations(all_elements, i):
        ...
```

This tries **every combination of every divisor size** as a candidate subalgebra seed. For A₅ (order 60), the `i=30` term alone is `C(60, 30) ≈ 1.2 × 10^17` combinations. I ran this and it didn't finish in any reasonable time — this is why `about_subalgebras` looks "slow": it's actually combinatorially infeasible, not just slow.

**Fix:** build the subalgebra lattice bottom-up instead of testing every subset. Any closed subset `H` containing generators `g1, ..., gk` is reachable by adding those generators one at a time and closing at each step: `closure({g1}) ⊆ closure({g1,g2}) ⊆ ... ⊆ H`, and each intermediate closure is itself a valid closed subalgebra contained in `H`. So a breadth-first search — start from every single element's closure, then repeatedly extend each discovered closed set by one more element and re-close, tracking what's already been found — discovers every closed proper subset without ever enumerating subsets explicitly. This is essentially the standard "cyclic extension" algorithm used for subgroup-lattice computation (Neubüser, 1960; also in Holt/Eick/O'Brien's *Handbook of Computational Group Theory*, and what GAP's `LatticeSubgroups` is based on).

```python
def closed_subsets_of_elements(self, divisors_only=True, include_inverses=True):
    """Return all unique, closed, proper subsets of the algebra's elements.
    This returns a list of lists. Each list represents the elements of a subalgebra.
    If divisors_only is True, only subalgebras of orders that are divisors of self's
    order are included in the result.

    Instead of testing every combination of elements of every candidate size (which is
    combinatorially infeasible -- e.g. C(60, 30) ~ 1.2*10^17 for an order-60 algebra
    like A5), this builds up the lattice of closed subsets from the bottom: starting
    from the closure of each individual element, it repeatedly extends every closed
    set found so far by one more element and re-closes, discovering every closed proper
    subset that way. This works because any closed subset H containing elements
    g1, ..., gk is reachable by adding those elements to the empty set one at a time and
    closing at each step -- closure({g1}) subseteq closure({g1,g2}) subseteq ... subseteq H
    -- and every intermediate closure along the way is itself closed and contained in H,
    so it is one of the sets this search discovers on the way to finding H.

    Reference: this is the "cyclic extension" method for building a subgroup lattice;
    see J. Neubuser, "Untersuchungen des Untergruppenverbandes endlicher Gruppen auf
    einer programmgesteuerten Rechenanlage," Numerische Mathematik 2 (1960): 280-292,
    and D. Holt, B. Eick, E. O'Brien, Handbook of Computational Group Theory, Chapman &
    Hall/CRC, 2005 (subgroup lattice / cyclic extension algorithm).
    """
    all_elements = self.elements
    n = len(all_elements)

    known = set()
    queue = []
    for x in all_elements:
        clo = frozenset(self.closure([x], include_inverses))
        if len(clo) < n and clo not in known:
            known.add(clo)
            queue.append(clo)

    while queue:
        subset = queue.pop()
        for g in all_elements:
            if g in subset:
                continue
            extended = frozenset(self.closure(list(subset) + [g], include_inverses))
            if len(extended) < n and extended not in known:
                known.add(extended)
                queue.append(extended)

    if divisors_only:
        allowed_sizes = set(divisors(n, non_trivial=True))
        known = {s for s in known if len(s) in allowed_sizes}

    return list(map(list, known))
```

I validated this against the existing brute-force version on S3, Z4, and V4 (identical results), then ran it standalone on S4 (order 24, 28 proper non-trivial subgroups in 0.5s — matches known counts) and on **A5**: **57 proper non-trivial subalgebras found in ~26s**, breaking down as 15 order-2, 10 order-3, 5 order-4, 6 order-5, 10 order-6, 6 order-10, 5 order-12 — which exactly matches A5's known subgroup structure. (One side effect: this also fixes a pre-existing gap where order-1 subalgebras were never found at all, since the old code never tried combination-size 1.)

## Bottleneck #2: `partition_into_isomorphic_lists` uses the slow `isomorphic`, not `fast_isomorphic`

```python
if gp.isomorphic(g):   # brute-force: tries other.order! permutations
```

You already have `fast_isomorphic` on `Magma` (from our earlier session) — it's just not being used here. For A5's order-12 subgroups (A4), `isomorphic` would try up to `12! ≈ 479 million` permutations *per comparison*. Swapping in `fast_isomorphic` is a straightforward, large win. I also fixed a small inefficiency: the recursive helper was comparing each candidate against itself (`rest = remainder` instead of `remainder[1:]`).

```python
def partition_into_isomorphic_lists(list_of_groups):
    """Partition the list of groups into sub-lists of groups that are isomorphic to each other.
    The purpose of this function is to operate on the proper subgroups of a group to determine
    the unique subgroups, up to isomorphism.

    Uses 'fast_isomorphic' rather than the brute-force 'isomorphic', since the latter tries
    every one of other.order! bijections and is intractable once subalgebra orders climb
    past roughly 8-9 (e.g. the order-12 subgroups of A5).
    """
    def iso_and_not_iso(gp, gps):
        """Partition the list of groups, gps, into two lists, those that are isomorphic to gp
        and those that are not."""
        iso_to_grp = []
        not_iso_to_grp = []
        for g in gps:
            if gp.fast_isomorphic(g):
                iso_to_grp.append(g)
            else:
                not_iso_to_grp.append(g)
        return iso_to_grp, not_iso_to_grp

    def aux(result, remainder):
        """Recursively partition 'remainder' into lists that are isomorphic to its first member
        and those that are not, then recurse on the remainder."""
        if len(remainder) == 0:
            return result
        else:
            first = remainder[0]
            rest = remainder[1:]  # was 'remainder', which wastefully compared 'first' to itself
            iso_to_first, not_iso_to_first = iso_and_not_iso(first, rest)
            result.append([first] + iso_to_first)
            return aux(result, not_iso_to_first)

    return aux([], list_of_groups)
```

Make the same substitution (`algebra.isomorphic(part[0])` → `algebra.fast_isomorphic(part[0])`) in `find_isomorphic_subalgebra`, further down in the same file.

## Result on A5

With both fixes: **subgroup enumeration ~14–26s** (down from never finishing), **partitioning 0.08s** (down from what would have been unusably slow given the order-12 subgroups). Partitioning correctly finds **7 isomorphism classes** — Z2, Z3, V4, Z5, S3, D5, A4 — matching A5's known subgroup structure exactly.

## Optional bonus (small, separate file)

While profiling I noticed `FiniteOperator._binary_operation` (in `finite_operator.py`) looks up each element's position with `self._elements.index(elem)` — an O(n) linear scan on every single operation call, including deep inside `closure()`. Replacing it with a dict built once in `__init__` cut the subgroup-enumeration time roughly in half (26s → 14s) in my testing, and speeds up every other method that calls `self.op` too:

```python
# In FiniteOperator.__init__, add:
self._index = {e: i for i, e in enumerate(elements)}

# Replace _binary_operation with:
def _binary_operation(self, elem1, elem2):
    row = self._index[elem1]
    col = self._index[elem2]
    return self._elements[self._table[row, col]]
```

This one's outside the scope of what you asked about, so I didn't want to just drop it in unannounced — let me know if you'd like it written up the same way as the other two.

## Prompt

Yes, nice catch. Please write it up the same way as the other two.

## Answer

Here's the writeup for the `FiniteOperator` fix, in the same format as the other two.

## Bottleneck #3: `FiniteOperator._binary_operation` does a linear scan on every call

```python
def _binary_operation(self, elem1, elem2):
    row = self._elements.index(elem1)   # O(n) linear scan
    col = self._elements.index(elem2)   # O(n) linear scan
    index = self._table[row, col]
    return self._elements[index]
```

Every call to an algebra's operation — `self.op(a, b)` — does two `list.index()` calls to translate element names into table positions, and `list.index()` scans the list from the front until it finds a match: O(n) per lookup, done twice per operation call. This isn't just `about_subalgebras`'s problem — it's a cost every method built on top of `op` pays, including the `closure()` calls at the heart of the new `closed_subsets_of_elements` from bottleneck #1: each closure computation does O(k²) operation calls while growing a k-element subset, and each of those now costs an extra O(n) on top just to find the two operands, i.e. this makes what should be O(k²) work cost O(k²·n).

**Fix:** build a name→index dict once, in `__init__`, and use it for O(1) lookup instead of re-scanning the element list on every call:

```python
class FiniteOperator:
    """A callable class that implements a binary operation based on a multiplication
    table (i.e., Cayley table). Although intended for use as the binary operation
    of a finite algebra (e.g., Group operation), the implementation here can be called
    with zero, one, two, or more arguments (similar to how arithmetic operators work in
    Lisp).

    If no arguments are provided, it will return the identity element if it exists;
    otherwise it will return None.  e.g., op() ==> e | None

    If only one argument is provided, it will check whether the argument is a valid
    element of the algebra, and if so, return the same value, otherwise it will
    raise an exception.  e.g., op(a) ==> a | ValueError

    If two arguments are provided, it will return their 'product'.
    e.g., op(a, b) ==> ab

    If more than two arguments are provided, it will return their product by associating
    left-to-right. e.g., op(a, b, c, d) = (((ab)c)d). The order of association is
    only important for a Magma, because it is the only non-associative algebraic structure
    supported here.
    """

    def __init__(self, elements, identity, table):
        self._elements = elements
        self._identity = identity
        self._table = table
        # Name -> position lookup, built once, so _binary_operation doesn't have to
        # linearly re-scan self._elements (via list.index) on every single call. This
        # matters a lot for methods like Magma.closure(), which call the operation
        # O(k^2) times while growing a k-element subset -- each avoided O(n) scan adds
        # up fast once both k and n are non-trivial (e.g. subalgebra enumeration on A5).
        self._index = {e: i for i, e in enumerate(elements)}

    def __call__(self, *args):
        return self._op(*args)

    def _binary_operation(self, elem1, elem2):
        """Returns the 'sum' of exactly two elements."""
        try:
            row = self._index[elem1]
            col = self._index[elem2]
        except KeyError as exc:
            bad = elem1 if elem1 not in self._index else elem2
            raise ValueError(f"{bad} is not a valid element name") from exc
        index = self._table[row, col]
        return self._elements[index]

    def _op(self, *args):
        if len(args) == 0:
            return self._identity
        elif len(args) == 1:
            if args[0] in self._elements:
                return args[0]
            else:
                raise ValueError(f"{args[0]} is not a valid element name")
        elif len(args) == 2:
            return self._binary_operation(args[0], args[1])
        else:
            return reduce(lambda a, b: self._op(a, b), args)
```

The `try`/`except` around the dict lookups preserves the original behavior of raising `ValueError` (not `KeyError`) for an invalid element name in the two-argument case, matching the error type the one-argument branch already raises — so if any caller catches `ValueError` from `op(a, b)`, that still works.

I measured this in isolation against the new `closed_subsets_of_elements` on A5: **~26s → ~14s**, roughly a 2x additional speedup on top of the subgroup-enumeration fix, since it's cutting cost out of `closure()`'s inner loop specifically. It also speeds up `fast_isomorphic`, `is_associative`, and everything else in the library that goes through `self.op` — the dict is built once per algebra and reused for its whole lifetime, so there's no downside.